# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SUKRIT004/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The playbook combines two validated signals into one ranked queue: the Week-4 transparent baseline (CTR below what a page's position tier typically gets, weighted by exposure) and the Week-5 Random Forest's decline-risk probability. Both are rescaled to 0–100 and combined 60/40 in favor of the baseline, since the held-out evaluation (ML-08/ML-09) showed the baseline meets or beats the model at Precision@50 (0.84 vs 0.74). The model score is kept as a secondary, corroborating signal rather than dropped — a page that both scores highly on CTR gap *and* on model-flagged decline risk is a stronger candidate than either alone. Every row carries plain-language reason codes (e.g. `low_ctr_vs_position`, `model_flags_decline_risk`, `high_exposure`, `stale_content`) so a reviewer can see *why* a page was prioritized without trusting a black box.

In [1]:
# ML-10 Section 1 — Build the ranked action queue
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

url = "https://raw.githubusercontent.com/SUKRIT004/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
data = pd.read_csv(url)
data["is_declining_label"] = (data["trend_direction"] == "down").astype(int)

visible = data[(data["impressions_90d"] > 0) & (data["avg_position"] > 0)].copy()

# --- Re-fit the Week-5 Random Forest (same features/split/seed as w05_model.ipynb) ---
target = "is_declining_label"
group = "client_id"
numeric_features = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "search_volume", "competition", "cpc",
]
categorical_features = [
    "content_type", "main_intent", "competition_level", "age_tier",
    "freshness_tier", "position_tier", "impression_tier",
]
features = numeric_features + categorical_features
X = data[features].copy()
y = data[target].astype(int)
groups = data[group]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])
model = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_leaf=10,
    class_weight="balanced", random_state=42, n_jobs=-1,
)
pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])
pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])

full_probability = pipeline.predict_proba(X)[:, 1]

# --- Baseline opportunity score (same rule as ML-07 / w04_baseline_score.ipynb) ---
playbook = visible.copy()
playbook["model_probability"] = pd.Series(full_probability, index=data.index).loc[playbook.index]

expected_ctr = playbook.groupby("position_tier")["ctr"].mean().rename("expected_ctr")
playbook = playbook.join(expected_ctr, on="position_tier")
playbook["ctr_gap"] = (playbook["expected_ctr"] - playbook["ctr"]).clip(lower=0)
playbook["baseline_score"] = playbook["ctr_gap"] * np.log1p(playbook["impressions_90d"])

def to_100(s):
    s = s.astype(float)
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) * 100 if hi > lo else s * 0

playbook["baseline_score_100"] = to_100(playbook["baseline_score"])
playbook["model_score_100"] = to_100(playbook["model_probability"])
playbook["final_score"] = 0.6 * playbook["baseline_score_100"] + 0.4 * playbook["model_score_100"]

def reason_codes(row):
    codes = []
    if row["ctr_gap"] > 0:
        codes.append("low_ctr_vs_position")
    if row["model_probability"] >= 0.6:
        codes.append("model_flags_decline_risk")
    if row["impressions_90d"] >= playbook["impressions_90d"].quantile(0.75):
        codes.append("high_exposure")
    if row["days_since_last_update"] >= 180:
        codes.append("stale_content")
    return "|".join(codes) if codes else "no_strong_signal"

playbook["reason_codes"] = playbook.apply(reason_codes, axis=1)

def suggested_action(row):
    if row["ctr_gap"] > 0 and row["model_probability"] >= 0.6:
        return "review_ctr_and_content"
    if row["ctr_gap"] > 0:
        return "review_ctr"
    if row["model_probability"] >= 0.6:
        return "monitor_decline_risk"
    return "no_action"

playbook["suggested_action"] = playbook.apply(suggested_action, axis=1)
playbook = playbook.sort_values("final_score", ascending=False).reset_index(drop=True)
playbook["rank"] = np.arange(1, len(playbook) + 1)

playbook_cols = [
    "rank", "content_type", "main_intent", "position_tier", "impression_tier",
    "age_tier", "freshness_tier", "impressions_90d", "ctr", "avg_position",
    "baseline_score_100", "model_score_100", "final_score",
    "reason_codes", "suggested_action",
]
action_playbook = playbook[playbook_cols].copy()

print("Ranked queue rows:", len(action_playbook))
display(action_playbook.head(15))


Ranked queue rows: 28795


,rank,content_type,main_intent,position_tier,impression_tier,age_tier,freshness_tier,impressions_90d,ctr,avg_position,baseline_score_100,model_score_100,final_score,reason_codes,suggested_action
0,1,keyword article,informational,top_3,good,181-365,0-30,24784,0.06,1.6,79.647932,86.906224,82.551249,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
1,2,keyword article,transactional,top_3,good,91-180,91-180,24260,0.08,2.0,78.891952,81.434637,79.909026,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
2,3,keyword article,informational,top_3,excellent,91-180,91-180,128068,0.01,2.2,94.287760,58.000078,79.772687,low_ctr_vs_position|high_exposure,review_ctr
3,4,keyword article,informational,top_3,good,181-365,0-30,16512,0.07,1.0,76.168536,84.323838,79.430657,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
4,5,keyword article,informational,top_3,good,91-180,0-30,8305,0.06,2.9,71.041884,91.683601,79.298571,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
5,6,keyword article,informational,top_3,good,91-180,0-30,29747,0.07,1.2,80.784919,76.934794,79.244869,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
6,7,keyword article,informational,top_3,good,91-180,0-30,26470,0.05,0.7,80.462413,76.713800,78.962968,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
7,8,keyword article,informational,top_3,good,91-180,0-30,12053,0.02,2.6,75.067628,82.184268,77.914284,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
8,9,keyword article,informational,top_3,good,91-180,0-30,19035,0.07,2.2,77.283657,78.817534,77.897208,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
9,10,keyword article,informational,top_3,good,91-180,0-30,10277,0.06,2.2,72.718818,85.408900,77.794851,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended user:** a content/SEO reviewer deciding which visible pages to look at first this week. **Intended use:** a triage aid — the queue narrows thousands of pages to a short, reasoned shortlist; a human still opens each page and decides. **Not intended for:** fully automated publishing changes, proof that any specific page will improve if edited, or any claim about Google's ranking algorithm. The queue only covers *visible* pages (impressions_90d > 0 and avg_position > 0, n={n:,}); pages with zero search visibility are out of scope for a CTR-review queue by definition. The `final_score` is a decision-support ranking, not a probability or a guarantee — language throughout stays observed / measured / directional per `skills/writing-honest-claims/SKILL.md`.

In [2]:
# ML-10 Section 2 — Scope check
print("Rows in queue (visible pages only):", len(action_playbook))
print("Rows excluded as not visible:", len(data) - len(action_playbook))
print("\nAction mix (what the queue recommends):")
print(action_playbook["suggested_action"].value_counts())


Rows in queue (visible pages only): 28795
Rows excluded as not visible: 1205

Action mix (what the queue recommends):
suggested_action
review_ctr                13591
review_ctr_and_content    10180
no_action                  3684
monitor_decline_risk       1340
Name: count, dtype: int64


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on any row: (1) open the page and confirm the CTR gap isn't explained by query intent mismatch (e.g. a page ranking for a query it doesn't actually answer well); (2) check whether the page is mid-migration, seasonal, or recently changed — those explain CTR dips without needing a rewrite; (3) confirm the position-tier average used as "expected CTR" is a reasonable comparison for this page's specific query type, not just its bucket. **Never automate:** publishing a content change from this queue alone, pausing/deprioritizing a page because of a low rank, or treating `model_flags_decline_risk` as a diagnosis of *why* a page is declining — the model gives no causal reason, only a correlation-based flag.

In [3]:
# ML-10 Section 3 — No-go / low-confidence flag for the reviewer
# Flag rows where exposure is thin — a small-impressions page can produce a large
# opportunity_score off very few observations, which is worth a reviewer's skepticism.
low_exposure_cutoff = action_playbook["impressions_90d"].quantile(0.10)
action_playbook["reviewer_flag"] = np.where(
    action_playbook["impressions_90d"] <= low_exposure_cutoff,
    "low_exposure_review_with_caution",
    "standard_review"
)
print("Low-exposure cutoff (impressions_90d):", round(low_exposure_cutoff, 1))
print(action_playbook["reviewer_flag"].value_counts())


Low-exposure cutoff (impressions_90d): 12.0
reviewer_flag
standard_review                     25870
low_exposure_review_with_caution     2925
Name: count, dtype: int64


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Retrain or re-audit the baseline/model when any of: (1) the position-tier CTR averages the baseline relies on shift meaningfully quarter over quarter (a sign query behavior or SERP layout changed); (2) the Random Forest's ROC-AUC on a fresh held-out client split drops materially below the ~0.61 observed here (a sign the relationships it learned no longer hold); (3) the action mix (`review_ctr` vs `review_ctr_and_content` vs `no_action`) shifts sharply between runs on comparable data, which usually means an upstream data or label change rather than a real trend; (4) six months pass, as a default cadence regardless of drift signals.

In [4]:
# ML-10 Section 4 — Snapshot the monitoring baseline to compare future runs against
monitoring_snapshot = {
    "expected_ctr_by_position_tier": (
        playbook.groupby("position_tier")["ctr"].mean().round(4).to_dict()
    ),
    "model_roc_auc_reference": 0.6145,
    "action_mix_reference": action_playbook["suggested_action"].value_counts().to_dict(),
}
print("Monitoring snapshot (compare future runs against this):")
for k, v in monitoring_snapshot.items():
    print(f"  {k}: {v}")


Monitoring snapshot (compare future runs against this):
  expected_ctr_by_position_tier: {'deep': 0.1502, 'page_1': 0.6525, 'page_3_5': 0.2225, 'striking': 0.3232, 'top_3': 2.7645}
  model_roc_auc_reference: 0.6145
  action_mix_reference: {'review_ctr': 13591, 'review_ctr_and_content': 10180, 'no_action': 3684, 'monitor_decline_risk': 1340}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Exports the full ranked queue (public-safe: no `content_id`/`client_id`, no client names, no URLs — only content-type/signal columns) to `work/outputs/action_playbook.csv`, plus a 20-row sample for quick inspection in the paper.

In [5]:
# ML-10 Section 5 — Export
from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

action_playbook.to_csv(output_dir / "action_playbook.csv", index=False)
action_playbook.head(20).to_csv(output_dir / "action_playbook_top20_sample.csv", index=False)

print("Wrote:", output_dir / "action_playbook.csv", "-", len(action_playbook), "rows")
print("Wrote:", output_dir / "action_playbook_top20_sample.csv", "- 20 rows")

# Public-safety check: confirm no identifier columns leaked into the export
forbidden = ["content_id", "client_id"]
leaked = [c for c in forbidden if c in action_playbook.columns]
assert not leaked, f"Forbidden identifier columns present: {leaked}"
print("\nPublic-safety check passed: no content_id/client_id in the export.")


Wrote: work/outputs/action_playbook.csv - 28795 rows
Wrote: work/outputs/action_playbook_top20_sample.csv - 20 rows

Public-safety check passed: no content_id/client_id in the export.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.